# HRD v2 Model Training Demo

Demonstrates the full v2 pipeline on **synthetic data**:
1. Generate realistic synthetic expression data with planted HRD signal
2. Apply tiered labeling and soft labels
3. Run confident learning to detect label noise
4. Train multi-task gradient boosting model
5. Compare against baselines (ElasticNet, Centroid, Single-gene)
6. Visualize: ROC curves, feature importance, noise detection

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.model_selection import train_test_split

np.random.seed(42)

## 1. Generate Synthetic Data

Create a dataset that mimics TCGA-BRCA:
- 500 samples, 200 genes
- ~25% HRD-positive (reflecting TCGA proportions)
- Planted signal in HR-repair and alt-EJ pathway genes
- Some deliberately mislabeled samples to test confident learning

In [ ]:
def generate_synthetic_hrd_data(n_samples=500, n_genes=200, hrd_fraction=0.25,
                                 noise_fraction=0.05, random_state=42):
    """Generate synthetic expression data with planted HRD signal."""
    rng = np.random.RandomState(random_state)
    
    n_hrd = int(n_samples * hrd_fraction)
    n_hrp = n_samples - n_hrd
    
    # Gene names: mix of HRD-relevant and noise genes
    hrd_genes = ['BRCA1', 'BRCA2', 'RAD51', 'POLQ', 'PARP1', 'CHEK1', 'CHEK2',
                 'ATR', 'ATM', 'EXO1', 'FANCD2', 'FANCA', 'RAD51C', 'PALB2',
                 'BRIP1', 'BARD1', 'MRE11', 'NBN', 'RAD50', 'TP53BP1']
    n_signal = len(hrd_genes)
    noise_genes = [f'GENE_{i}' for i in range(n_genes - n_signal)]
    gene_names = hrd_genes + noise_genes
    
    # Base expression (log-normal)
    X = rng.lognormal(mean=3, sigma=1.5, size=(n_samples, n_genes))
    
    # Plant signal in HRD samples
    hrd_idx = np.arange(n_hrd)
    hrp_idx = np.arange(n_hrd, n_samples)
    
    # BRCA1 downregulated in ~40% of HRD (BRCA1-type)
    brca1_down = rng.choice(hrd_idx, size=int(n_hrd * 0.4), replace=False)
    X[brca1_down, 0] *= 0.15  # BRCA1
    
    # BRCA2 downregulated in ~30% of HRD (BRCA2-type)
    brca2_down = np.setdiff1d(hrd_idx, brca1_down)[:int(n_hrd * 0.3)]
    X[brca2_down, 1] *= 0.2  # BRCA2
    
    # POLQ upregulated in HRD (alt-EJ compensation)
    X[hrd_idx, 3] *= 2.5  # POLQ
    
    # RAD51 loading reduced in HRD
    X[hrd_idx, 2] *= 0.6  # RAD51
    
    # General DDR upregulation in HRD
    for g in [4, 5, 6, 7, 10, 11]:  # PARP1, CHEK1, CHEK2, ATR, FANCD2, FANCA
        X[hrd_idx, g] *= rng.uniform(1.3, 1.8)
    
    # Labels
    y_true = np.zeros(n_samples, dtype=int)
    y_true[:n_hrd] = 1
    
    # Simulate HRD-sum scores (correlated with true label)
    hrd_sum = np.zeros(n_samples)
    hrd_sum[hrd_idx] = rng.normal(55, 15, size=n_hrd)  # HRD samples: high GIS
    hrd_sum[hrp_idx] = rng.normal(12, 8, size=n_hrp)   # HRP samples: low GIS
    hrd_sum = np.clip(hrd_sum, 0, 100).astype(int)
    
    # Introduce label noise (mislabel some samples)
    n_noisy = int(n_samples * noise_fraction)
    noisy_idx = rng.choice(n_samples, size=n_noisy, replace=False)
    y_noisy = y_true.copy()
    y_noisy[noisy_idx] = 1 - y_noisy[noisy_idx]
    
    # BRCA type labels
    brca_type = np.full(n_samples, 'HRP', dtype=object)
    brca_type[brca1_down] = 'BRCA1'
    brca_type[brca2_down] = 'BRCA2'
    remaining_hrd = np.setdiff1d(hrd_idx, np.concatenate([brca1_down, brca2_down]))
    brca_type[remaining_hrd] = 'HRD_BRCApos'
    
    # Sample IDs
    sample_ids = [f'SAMPLE-{i:04d}' for i in range(n_samples)]
    
    # Build DataFrames
    expr_df = pd.DataFrame(X, index=sample_ids, columns=gene_names)
    
    metadata = pd.DataFrame({
        'HRD-sum': hrd_sum,
        'true_label': y_true,
        'noisy_label': y_noisy,
        'brca_type': brca_type,
        'is_noisy': False,
    }, index=sample_ids)
    metadata.loc[metadata.index[noisy_idx], 'is_noisy'] = True
    
    return expr_df, metadata, noisy_idx


expr_df, metadata, true_noisy_idx = generate_synthetic_hrd_data()
print(f"Expression: {expr_df.shape}")
print(f"Metadata: {metadata.shape}")
print(f"True HRD: {(metadata['true_label'] == 1).sum()}, HRP: {(metadata['true_label'] == 0).sum()}")
print(f"Noisy labels: {metadata['is_noisy'].sum()}")

## 2. Label Smoothing

Convert hard binary labels to soft targets using HRD-sum scores.

In [ ]:
from v2.models.label_smoothing import soft_label_from_score, smooth_binary_labels, apply_label_smoothing

# Method 1: Sigmoid mapping from HRD-sum
soft_sigmoid = soft_label_from_score(metadata['HRD-sum'], method='sigmoid')

# Method 2: Linear mapping
soft_linear = soft_label_from_score(metadata['HRD-sum'], method='linear')

# Method 3: Blend hard + continuous
soft_blended = smooth_binary_labels(metadata['noisy_label'], metadata['HRD-sum'], temperature=0.7)

# Method 4: Uniform smoothing
soft_uniform = apply_label_smoothing(metadata['noisy_label'], alpha=0.1)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, soft, title in zip(axes.flat,
    [soft_sigmoid, soft_linear, soft_blended, soft_uniform],
    ['Sigmoid', 'Linear', 'Blended (temp=0.7)', 'Uniform (alpha=0.1)']):
    ax.scatter(metadata['HRD-sum'], soft, c=metadata['true_label'],
              cmap='RdBu_r', alpha=0.5, s=15)
    ax.set_xlabel('HRD-sum')
    ax.set_ylabel('Soft label')
    ax.set_title(f'Soft labels: {title}')
    ax.axvline(42, color='gray', linestyle='--', alpha=0.5, label='GIS threshold')
    ax.legend()

plt.tight_layout()
plt.show()

## 3. Confident Learning

Detect mislabeled samples before training.

In [ ]:
from v2.models.confident_learning import ConfidentLearner

cl = ConfidentLearner(n_folds=5, calibration_method='isotonic')

# Use noisy labels (simulating what we'd get from simple threshold)
results = cl.find_noisy_labels(expr_df.values, metadata['noisy_label'].values)

print(f"Flagged as noisy: {results['noise_mask'].sum()}")
print(f"Actually noisy: {metadata['is_noisy'].sum()}")

# Check overlap with true noisy samples
detected_noisy = set(np.where(results['noise_mask'])[0])
actual_noisy = set(true_noisy_idx)
overlap = detected_noisy & actual_noisy

print(f"\nTrue positives (correctly detected): {len(overlap)}")
print(f"False positives (wrongly flagged): {len(detected_noisy - actual_noisy)}")
print(f"False negatives (missed): {len(actual_noisy - detected_noisy)}")

if actual_noisy:
    recall = len(overlap) / len(actual_noisy)
    precision = len(overlap) / len(detected_noisy) if detected_noisy else 0
    print(f"Recall: {recall:.2f}, Precision: {precision:.2f}")

In [ ]:
# Visualize label quality
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Label quality distribution
ax = axes[0]
ax.hist(results['label_quality'][~metadata['is_noisy'].values], bins=30,
        alpha=0.6, label='Clean samples', color='steelblue')
ax.hist(results['label_quality'][metadata['is_noisy'].values], bins=30,
        alpha=0.6, label='Actually noisy', color='red')
ax.axvline(results['threshold_used'], color='black', linestyle='--',
           label=f'Threshold ({results["threshold_used"]:.2f})')
ax.set_xlabel('Label quality score')
ax.set_ylabel('Count')
ax.set_title('Label Quality Distribution')
ax.legend()

# Predicted prob vs HRD-sum
ax = axes[1]
scatter = ax.scatter(metadata['HRD-sum'], results['predicted_probs'],
                     c=metadata['is_noisy'].astype(int), cmap='RdBu_r',
                     alpha=0.5, s=15)
ax.set_xlabel('HRD-sum')
ax.set_ylabel('Predicted P(HRD)')
ax.set_title('Model predictions vs HRD-sum (red = noisy)')
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
ax.axvline(42, color='gray', linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()

## 4. Multi-Task Model Training

Train XGBoost models for HRD status + BRCA subtype simultaneously.

In [ ]:
from v2.models.multi_task_model import MultiTaskHRDModel
from sklearn.preprocessing import LabelEncoder

# Prepare labels
y_dict = {
    'hrd_status': soft_sigmoid,  # soft labels
    'brca_type': LabelEncoder().fit_transform(metadata['brca_type']),
}

# Get sample weights from confident learning
_, _, sample_weights = cl.clean_dataset(expr_df.values, metadata['noisy_label'].values, strategy='weight')

# Train/test split
train_idx, test_idx = train_test_split(np.arange(len(expr_df)), test_size=0.2, random_state=42,
                                        stratify=metadata['true_label'])

X_train = expr_df.iloc[train_idx]
X_test = expr_df.iloc[test_idx]
y_train = {k: v[train_idx] for k, v in y_dict.items()}
y_test = {k: v[test_idx] for k, v in y_dict.items()}
sw_train = sample_weights[train_idx]

# Build and train model
model = MultiTaskHRDModel(
    tasks=['hrd_status', 'brca_type'],
    base_model='xgboost',
    use_label_smoothing=True,
)

model.fit(X_train, y_train, sample_weights=sw_train)

# Predictions
preds = model.predict(X_test)
probas = model.predict_proba(X_test)

print("Tasks fitted:", list(model.models_.keys()))
print("Task types:", model.task_types_)

In [ ]:
# Evaluate HRD prediction
y_true_binary = metadata['true_label'].values[test_idx]

# For regression task, predictions are continuous
hrd_preds = probas.get('hrd_status', preds['hrd_status'])
if hrd_preds.ndim > 1:
    hrd_preds = hrd_preds[:, -1]

hrd_auc = roc_auc_score(y_true_binary, hrd_preds)
print(f"Multi-task model HRD AUC: {hrd_auc:.3f}")

## 5. Nested Cross-Validation

In [ ]:
# Full nested CV
cv_model = MultiTaskHRDModel(
    tasks=['hrd_status', 'brca_type'],
    base_model='xgboost',
    use_label_smoothing=True,
)

cv_results = cv_model.cross_validate(
    expr_df, y_dict,
    cv_strategy='stratified',
    n_splits=5,
    sample_weights=sample_weights,
)

for task_name, metrics in cv_results.items():
    print(f"\n--- {task_name} ---")
    for k, v in metrics.items():
        if 'mean' in k:
            std_key = k.replace('mean', 'std')
            std = metrics.get(std_key, 0)
            print(f"  {k}: {v:.3f} +/- {std:.3f}")

## 6. Baseline Comparisons

In [ ]:
from v2.models.baseline_models import ElasticNetBaseline, CentroidBaseline, SingleGeneBaseline

y_binary_noisy = metadata['noisy_label'].values

# -- ElasticNet (v1 approach) --
enet = ElasticNetBaseline(mode='classification', alpha=0.1, l1_ratio=0.5)
enet.fit(expr_df.values[train_idx], y_binary_noisy[train_idx])
enet_proba = enet.predict_proba(expr_df.values[test_idx])[:, 1]
enet_auc = roc_auc_score(y_true_binary, enet_proba)

# -- Centroid --
centroid = CentroidBaseline(n_genes=50)
centroid.fit(expr_df.values[train_idx], y_binary_noisy[train_idx])
cent_proba = centroid.predict_proba(expr_df.values[test_idx])[:, 1]
cent_auc = roc_auc_score(y_true_binary, cent_proba)

# -- Single Gene --
single = SingleGeneBaseline(gene_names=['POLQ', 'BRCA1', 'BRCA2'])
single.fit(expr_df, y_binary_noisy, feature_names=list(expr_df.columns))
sg_proba = single.predict_proba(expr_df.iloc[test_idx])[:, 1]
sg_auc = roc_auc_score(y_true_binary, sg_proba)

print(f"Multi-task XGBoost AUC:   {hrd_auc:.3f}")
print(f"ElasticNet (v1) AUC:      {enet_auc:.3f}")
print(f"Centroid AUC:             {cent_auc:.3f}")
print(f"Single gene ({single.best_gene_}) AUC: {sg_auc:.3f}")
print(f"\nSingle gene per-gene AUCs:")
print(single.summary())

In [ ]:
# ROC curves comparison
fig, ax = plt.subplots(figsize=(8, 7))

models_for_roc = [
    ('Multi-task XGBoost', hrd_preds),
    ('ElasticNet (v1)', enet_proba),
    ('Centroid', cent_proba),
    (f'Single gene ({single.best_gene_})', sg_proba),
]

colors = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0']

for (name, proba), color in zip(models_for_roc, colors):
    fpr, tpr, _ = roc_curve(y_true_binary, proba)
    auc_val = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc_val:.3f})', color=color, linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Comparison: Multi-task XGBoost vs Baselines', fontsize=14)
ax.legend(fontsize=11)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])

plt.tight_layout()
plt.show()

## 7. Feature Importance (Gain-based)

SHAP analysis requires the `shap` package. Here we show gain-based importance which works out-of-the-box.

In [ ]:
# Feature importance
importance_df = model.get_feature_importance(method='gain')

# Top 20 features for HRD task
hrd_imp = importance_df[importance_df['task'] == 'hrd_status'].head(20)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# HRD task
ax = axes[0]
ax.barh(range(len(hrd_imp)), hrd_imp['importance'].values, color='steelblue')
ax.set_yticks(range(len(hrd_imp)))
ax.set_yticklabels(hrd_imp['feature'].values)
ax.set_xlabel('Feature importance (gain)')
ax.set_title('Top 20 features: HRD status')
ax.invert_yaxis()

# BRCA type task
brca_imp = importance_df[importance_df['task'] == 'brca_type'].head(20)
ax = axes[1]
ax.barh(range(len(brca_imp)), brca_imp['importance'].values, color='coral')
ax.set_yticks(range(len(brca_imp)))
ax.set_yticklabels(brca_imp['feature'].values)
ax.set_xlabel('Feature importance (gain)')
ax.set_title('Top 20 features: BRCA type')
ax.invert_yaxis()

plt.tight_layout()
plt.show()

# Check: do HRD-relevant genes dominate?
hrd_gene_names = {'BRCA1', 'BRCA2', 'RAD51', 'POLQ', 'PARP1', 'CHEK1', 'CHEK2',
                  'ATR', 'ATM', 'EXO1', 'FANCD2', 'FANCA'}
top_10_genes = set(hrd_imp.head(10)['feature'])
overlap = top_10_genes & hrd_gene_names
print(f"\nTop 10 HRD features that are known HRD genes: {len(overlap)}/10")
print(f"Genes: {overlap}")

## 8. SHAP Analysis (if available)

In [ ]:
try:
    import shap
    
    # SHAP for HRD task
    hrd_model = model.models_['hrd_status']
    explainer = shap.TreeExplainer(hrd_model)
    shap_values = explainer.shap_values(X_test.values[:100])
    
    fig, ax = plt.subplots(figsize=(10, 8))
    shap.summary_plot(shap_values, X_test.iloc[:100],
                     feature_names=list(X_test.columns),
                     max_display=20, show=False)
    plt.title('SHAP values: HRD status prediction')
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print("shap not installed. Install with: pip install shap")
    print("Skipping SHAP analysis.")

## 9. Summary

Key observations from this synthetic demo:
- Confident learning correctly identifies a majority of mislabeled samples
- Soft labels (sigmoid mapping) produce better-calibrated probabilities than hard thresholds
- Multi-task XGBoost outperforms simple ElasticNet on this data
- The planted signal genes (POLQ, BRCA1, BRCA2, etc.) dominate feature importance

**Next steps for real data:**
1. Use `HRDTrainingPipeline.run()` with actual TCGA expression + metadata
2. Enable leave-one-dataset-out CV with `dataset_col`
3. Run SHAP analysis for biological interpretation
4. Validate on held-out cohorts (I-SPY2, CCLE)

In [ ]:
# Quick demo of the full pipeline (using synthetic data without tiered labeling)
from v2.models.training_pipeline import HRDTrainingPipeline, FeatureConfig, ModelConfig, LabelConfig

pipeline = HRDTrainingPipeline(
    feature_config=FeatureConfig(
        use_rank_pairs=False,      # skip for synthetic data
        use_pathway_scores=False,  # skip for synthetic data  
        use_ratios=False,          # skip for synthetic data
        use_raw_expression=True,
    ),
    model_config=ModelConfig(
        base_model='xgboost',
        tasks=['hrd_status'],
        use_label_smoothing=True,
        cv_n_splits=5,
    ),
    label_config=LabelConfig(
        run_confident_learning=True,
        noise_strategy='weight',
        soft_label_method='sigmoid',
    ),
)

# Note: the pipeline expects metadata with 'HRD-sum' column
results = pipeline.run(
    expression_df=expr_df,
    metadata_df=metadata,
    run_baselines=True,
)

print("\n=== Pipeline Results ===")
print(f"Timing: {results['timing']}")
print(f"\nCV Results:")
for task, metrics in results['cv_results'].items():
    for k, v in metrics.items():
        if 'mean' in k:
            print(f"  {task}/{k}: {v:.3f}")